# SealedRead trên Colab

Chạy lần lượt từ trên xuống. Chi tiết: `docs/05-running.md`.

1. Chọn **Runtime → Change runtime type → A100 GPU**.
2. Các bước dài chạy nền và ghi log ra `/content/logs/`; ô cuối notebook để xem log.
3. Kết quả được chép sang Google Drive (`MyDrive/sealedread/`) mỗi 5 phút, nên runtime bị reset vẫn chạy tiếp được (marker trên Drive).
4. Làm theo `docs/07-colab-test-plan.md`: A (test CPU) → B (diễn tập 0.6B) → C (đo đạc 4B). S0 thật và huấn luyện chạy trên máy 8×B200.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Drive: shard gốc (dùng chung với ragproj2 / keychain-dv) và nơi lưu kết quả
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Lấy code (lần sau chạy lại ô này sẽ git pull bản mới nhất).
import os, subprocess
REPO_URL = 'github.com/thudenhasang/sealedread.git'   # repo public: không cần token
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = f'https://{token}@{REPO_URL}' if token else f'https://{REPO_URL}'
if not os.path.exists('/content/sealedread'):
    subprocess.run(['git', 'clone', '-q', url, '/content/sealedread'], check=True)
else:
    subprocess.run(['git', '-C', '/content/sealedread', 'pull', '-q'], check=True)
print(subprocess.run(['git', '-C', '/content/sealedread', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)

In [ ]:
# Môi trường (vLLM + torch + transformers trong một venv), vài phút; rồi chạy test CPU để chắc code không vỡ
!cd /content/sealedread && bash scripts/colab/setup.sh 2>&1 | tail -5
!cd /content/sealedread && /content/venv-srd/bin/python -m pytest -q tests 2>&1 | tail -3

In [ ]:
# Khôi phục dữ liệu/kết quả cũ từ Drive (nếu có) và bật đồng bộ lên Drive mỗi 5 phút
!cd /content/sealedread && bash scripts/colab/restore.sh
!cd /content/sealedread && nohup bash scripts/colab/persist.sh 300 > /content/logs/persist.log 2>&1 &

In [ ]:
# Hàm tiện ích: chạy một lệnh nền với log riêng, và xem log
import subprocess, pathlib
def bg(name, cmd):
    log = open(f'/content/logs/{name}.log', 'wb')
    p = subprocess.Popen(['bash', '-lc', f'cd /content/sealedread && {cmd}'], stdout=log, stderr=subprocess.STDOUT,
                         start_new_session=True)
    print('started', name, 'pid', p.pid)
def tail(name, n=30):
    p = pathlib.Path(f'/content/logs/{name}.log')
    print('\n'.join(p.read_text(errors='replace').splitlines()[-n:]) if p.exists() else '(no log yet)')

In [ ]:
# Giai đoạn A: chạy thử toàn chuỗi trên CPU với mô hình tí hon (khoảng 2 phút). Phải thấy SMOKE_DONE
!cd /content/sealedread && /content/venv-srd/bin/python scripts/smoke.py --out /content/runs/smoke 2>&1 | grep -E '^\[|SMOKE|Error' | tail -12

In [ ]:
# Giai đoạn B (docs/07-colab-test-plan.md): diễn tập toàn chuỗi với Qwen3-0.6B, ngữ cảnh 4-16K,
# dữ liệu từ shard train. Con số không có ý nghĩa; mọi script phải chạy hết không cần sửa tay.
bg('rehearsal', 'bash scripts/colab/run_rehearsal.sh')

In [ ]:
# Giai đoạn C: đo đạc với Qwen3-4B trên A100: tokenizer, dữ liệu thật (shard train), parity, chi phí,
# P0 thu nhỏ, teacher pilot (chỉ A100 80 GB). Điền kết quả vào docs/results/colab-test.md
bg('measure', 'bash scripts/colab/run_measure.sh')

In [ ]:
# Một lần SFT (chạy lại đúng lệnh để tiếp tục sau khi runtime reset)
# bg('sft_sr', 'bash scripts/colab/run_sft.sh configs/sft.yaml train.branch=sealedread run.name=sft_sealedread_s0')

In [ ]:
tail('rehearsal')   # hoặc tail('measure')

In [ ]:
# Xem kết quả giai đoạn C (chạy khi log 'measure' báo MEASURE_DONE, hoặc bất cứ lúc nào để xem phần đã xong)
import json, pathlib
R = pathlib.Path('/content/runs/measure')
def show(p, keys=None):
    p = pathlib.Path(p)
    if not p.exists():
        print('(chưa có)', p); return
    d = json.loads(p.read_text())
    print('==', p.relative_to(R)); print(json.dumps(d if keys is None else {k: d.get(k) for k in keys}, indent=1, ensure_ascii=False)[:2500])
show(R/'tokenizer_check.json', ['pass', 'max_locate_tokens', 'copy_length_mismatches', 'same_length_per_digit_count'])
if (R/'checks.json').exists():
    ck = json.loads((R/'checks.json').read_text())
    print('== checks.json'); print({f: ({k: v.get('pass') for k, v in r.items()} if 'pass' not in r else r) for f, r in ck.items()})
show(R/'parity.json')
show(R/'cost_64k/cost_probe_engine.json'); show(R/'cost_128k/cost_probe_engine.json')
for b in ('sealedread', 'nomask'):
    show(R/f'cost_sft_{b}/cost_probe_sft.json')
show(R/'s0/p0_pilot.json', ['delta_oracle', 'pi', 'pilot_reading', 'other'])
show(R/'s0/recall_at_k.json')
show(R/'teacher_pilot/pilot.json')